# Notebook 2: Variant Quality Control Exercise

**Scenario**: You are a bioinformatician analyzing variants from 200 whole-genome samples.
Your goal is to perform quality control and generate summary statistics using Spark.

**Estimated time**: 2-2.5 hours

**Instructions**: Fill in all cells marked with `YOUR CODE HERE`. Answer all questions
marked with `YOUR ANSWER:`.

In [ ]:
import os
import sys
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pandas as pd

# Add scripts directory to path
scripts_dir = "/workspace/scripts"
if not os.path.exists(scripts_dir):
    scripts_dir = os.path.join(os.path.dirname(os.getcwd()), "scripts")
sys.path.insert(0, scripts_dir)

from timing_utils import time_operation, compare_operations, partition_info
from viz_utils import plot_chromosome_distribution, plot_quality_histogram, plot_af_distribution

spark = SparkSession.builder \
    .appName("BINFX410_Variant_QC") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

data_dir = "/workspace/data"
if not os.path.exists(data_dir):
    data_dir = os.path.join(os.path.dirname(os.getcwd()), "data")

print(f"Spark UI: http://localhost:4040")
print("Setup complete!")

---
## Task 1: Load and Explore Data (~10 min)

Load the **medium** variant dataset (100,000 variants) and explore its structure.

In [ ]:
# TODO: Load the medium variant dataset (variants_medium.csv)
# Hint: Use spark.read.csv() with header=True, inferSchema=True

variants = # YOUR CODE HERE

# TODO: Display the schema using printSchema()
# YOUR CODE HERE

# TODO: Show the first 10 rows
# YOUR CODE HERE

# TODO: Count the total number of variants
# YOUR CODE HERE

In [ ]:
# TODO: Display summary statistics for QUAL, DP, AF, and GQ columns
# Hint: Use .select(...).summary().show()

# YOUR CODE HERE

# TODO: Count the number of distinct chromosomes
# Hint: Use .select("CHROM").distinct().count()

# YOUR CODE HERE

---
## Task 2: Basic Quality Filtering (~15 min)

Apply GATK-recommended hard filters to identify high-quality variants:
- **QUAL >= 30**: Variant quality score (Phred-scaled confidence)
- **DP >= 10**: Minimum read depth for reliable calls
- **GQ >= 20**: Minimum genotype quality

In [ ]:
# TODO: Apply all three quality filters
# Store the result in 'filtered_variants'

filtered_variants = # YOUR CODE HERE

# TODO: Count filtered variants and calculate the percentage that passed
total = variants.count()
passed = # YOUR CODE HERE

print(f"Total variants: {total}")
print(f"Passed filters: {passed}")
print(f"Filtered out: {total - passed} ({(total - passed) / total * 100:.1f}%)")

In [ ]:
# TODO: Investigate which individual filter removes the most variants

qual_pass = variants.filter(F.col("QUAL") >= 30).count()
dp_pass = # YOUR CODE HERE
gq_pass = # YOUR CODE HERE

print(f"Pass QUAL >= 30: {qual_pass} ({qual_pass/total*100:.1f}%)")
print(f"Pass DP >= 10:   {dp_pass} ({dp_pass/total*100:.1f}%)")
print(f"Pass GQ >= 20:   {gq_pass} ({gq_pass/total*100:.1f}%)")

# QUESTION: Which filter removes the most variants? Why might that be?
# YOUR ANSWER: 

---
## Task 3: Per-Chromosome Statistics (~20 min)

Calculate summary statistics for each chromosome.

In [ ]:
# TODO: For each chromosome, calculate:
#   - Total variant count
#   - Mean QUAL score
#   - Mean depth (DP)
#   - Number of PASS variants (where FILTER == "PASS")
#   - Pass rate (PASS count / total count)
#
# Hint: Use groupBy("CHROM").agg(...) with F.count, F.mean, F.sum, F.when
# Store result in 'chrom_stats'

chrom_stats = # YOUR CODE HERE

# TODO: Show the results sorted by variant_count descending
# YOUR CODE HERE

In [ ]:
# TODO: Convert chrom_stats to pandas and create a visualization
# Hint: Use .toPandas() and plot_chromosome_distribution()

chrom_stats_pd = # YOUR CODE HERE

# YOUR CODE HERE to create the plot

# QUESTION: Which chromosome has the highest variant count? Why?
# YOUR ANSWER: 

# QUESTION: Look at the Spark UI (Jobs tab) - how many stages did
# the groupBy aggregation have? Why?
# YOUR ANSWER: 

---
## Task 4: Allele Frequency Stratification (~20 min)

Categorize variants by allele frequency. In population genetics:
- **Singleton** (AF < 0.001): Seen in only one individual
- **Rare** (0.001-0.01): Present in <1% of the population
- **Low frequency** (0.01-0.05): Present in 1-5%
- **Common** (>= 0.05): Present in >5%

In [ ]:
# TODO: Create a new column called "frequency_class" based on AF:
#   - "Singleton":      AF < 0.001
#   - "Rare":           0.001 <= AF < 0.01
#   - "Low frequency":  0.01 <= AF < 0.05
#   - "Common":         AF >= 0.05
#
# Hint: Use F.when().when().when().otherwise()

variants_classified = # YOUR CODE HERE

# TODO: Count variants in each frequency class
# YOUR CODE HERE

In [ ]:
# TODO: Cross-tabulate frequency_class by chromosome
# Hint: Use groupBy("CHROM", "frequency_class").count() then pivot

freq_by_chrom = # YOUR CODE HERE

# YOUR CODE HERE to show results

# QUESTION: Are rare variants uniformly distributed across chromosomes?
# YOUR ANSWER: 

---
## Task 5: Join with Gene Annotations (~25 min)

Load gene annotations and identify which variants fall within genes.
A variant at position POS is within a gene if: `gene.start <= POS <= gene.end`
and the chromosomes match.

In [ ]:
# TODO: Load the gene annotations file (genes.bed)
# It's tab-separated with no header
# Schema: chrom STRING, start INT, end INT, gene_name STRING, strand STRING

genes = # YOUR CODE HERE

print(f"Loaded {genes.count()} gene annotations")
genes.show(5)

In [ ]:
# TODO: Join variants with genes based on position
# Conditions:
#   - variants.CHROM == genes.chrom
#   - genes.start <= variants.POS <= genes.end
# Use a LEFT join to keep all variants

variant_genes = # YOUR CODE HERE

# TODO: Count how many variants fall within genes
in_gene = # YOUR CODE HERE
total = variants.count()

print(f"Variants in genes: {in_gene}")
print(f"Intergenic variants: {total - in_gene}")
print(f"Percentage in genes: {in_gene/total*100:.1f}%")

In [ ]:
# TODO: Find the top 10 genes with the most variants
# Hint: Filter for non-null gene_name, groupBy, count, orderBy desc, limit 10

top_genes = # YOUR CODE HERE

# YOUR CODE HERE to show results

# QUESTION: What percentage of variants are intergenic?
# YOUR ANSWER: 

# QUESTION: Did this join cause a shuffle? Check the Spark UI (SQL tab).
# YOUR ANSWER: 

---
## Task 6: Performance Comparison - Spark vs pandas (~20 min)

Compare Spark and pandas performance on the same task.

In [ ]:
# Spark approach (cold - no caching)
start = time.time()
spark_result = variants.groupBy("CHROM").agg(
    F.count("*").alias("count"),
    F.mean("QUAL").alias("mean_qual")
).collect()
spark_time = time.time() - start
print(f"Spark time (cold): {spark_time:.3f}s")

# TODO: pandas approach - load the same CSV with pandas and do the same groupby
start = time.time()

pandas_df = # YOUR CODE HERE - load CSV with pd.read_csv
pandas_result = # YOUR CODE HERE - groupby CHROM, calculate count and mean QUAL

pandas_time = time.time() - start
print(f"Pandas time: {pandas_time:.3f}s")

In [ ]:
# TODO: Now try with the cached Spark DataFrame
variants.cache()
variants.count()  # trigger cache

start = time.time()
spark_cached_result = variants.groupBy("CHROM").agg(
    F.count("*").alias("count"),
    F.mean("QUAL").alias("mean_qual")
).collect()
spark_cached_time = time.time() - start

variants.unpersist()

print(f"Pandas:         {pandas_time:.3f}s")
print(f"Spark (cold):   {spark_time:.3f}s")
print(f"Spark (cached): {spark_cached_time:.3f}s")

# QUESTION: Which is faster on this 100K dataset? Why?
# YOUR ANSWER: 

# QUESTION: At what data size would Spark become necessary?
# YOUR ANSWER: 

# QUESTION: What is the overhead of using Spark?
# YOUR ANSWER: 

---
## Task 7: Optimization Challenge (~30 min)

The query below finds rare coding variants (in genes, AF < 0.01). Your goal is to
optimize it for better performance.

In [ ]:
# UNOPTIMIZED query
genes_for_join = spark.read.csv(
    os.path.join(data_dir, "genes.bed"),
    sep="\t", header=False,
    schema="chrom STRING, start INT, end INT, gene_name STRING, strand STRING"
)

start = time.time()
result_unoptimized = variants \
    .join(genes_for_join,
          (variants.CHROM == genes_for_join.chrom) &
          (variants.POS >= genes_for_join.start) &
          (variants.POS <= genes_for_join.end)) \
    .filter(F.col("AF") < 0.01) \
    .select("CHROM", "POS", "gene_name", "AF")

count_unopt = result_unoptimized.count()
time_unoptimized = time.time() - start

print(f"Unoptimized: {count_unopt} rare coding variants in {time_unoptimized:.3f}s")
print("\nExecution plan:")
result_unoptimized.explain()

In [ ]:
# TODO: Optimize this query! Try these strategies:
# 1. Filter AF < 0.01 BEFORE the join (reduce data shuffled)
# 2. Broadcast the genes table (it's small enough to send to all workers)
#    Hint: Use F.broadcast(genes_for_join)
# 3. Select only needed columns early

start = time.time()

result_optimized = # YOUR CODE HERE

count_opt = result_optimized.count()
time_optimized = time.time() - start

print(f"Optimized: {count_opt} rare coding variants in {time_optimized:.3f}s")
print(f"Speedup: {time_unoptimized/time_optimized:.1f}x")
print("\nOptimized execution plan:")
result_optimized.explain()

In [ ]:
# QUESTION: Which optimization helped the most? Why?
# YOUR ANSWER: 

# QUESTION: What does "BroadcastHashJoin" vs "SortMergeJoin" mean in the plan?
# YOUR ANSWER: 

---
## Task 8: Write Results (~10 min)

Save your filtered variants in **Parquet** format - the preferred format for big data
analytics.

In [ ]:
# TODO: Save filtered_variants as Parquet
output_path = os.path.join(data_dir, "filtered_variants.parquet")

# YOUR CODE HERE

print(f"Saved to {output_path}")

In [ ]:
# TODO: Compare file sizes
csv_path = os.path.join(data_dir, "variants_medium.csv")
csv_size = os.path.getsize(csv_path)

parquet_size = sum(
    os.path.getsize(os.path.join(dirpath, filename))
    for dirpath, _, filenames in os.walk(output_path)
    for filename in filenames
)

print(f"CSV size:     {csv_size / 1024 / 1024:.1f} MB")
print(f"Parquet size: {parquet_size / 1024 / 1024:.1f} MB")
print(f"Compression ratio: {csv_size / parquet_size:.1f}x")

# QUESTION: Why is Parquet preferred for big data?
# YOUR ANSWER: 

---
## Reflection Questions

Answer the following questions in 2-3 sentences each.

**1. When would you use Spark instead of pandas for genomics analysis?**

*YOUR ANSWER:*



**2. What are the main costs of distributed computing? (Hint: not just money)**

*YOUR ANSWER:*



**3. How would you design a Spark job to analyze 100,000 whole genomes?**

*YOUR ANSWER:*



**4. What did you learn from the Spark UI about how your queries executed?**

*YOUR ANSWER:*



In [ ]:
spark.stop()
print("Exercise complete! Don't forget to answer the reflection questions above.")